# RevIQ AI — Model Evaluation Notebook

**Purpose:** Evaluate the churn prediction models with the rigor expected in a real business context. We go beyond raw accuracy to understand *how* the models perform on the cases that matter most — identifying customers who will actually churn.

**Models evaluated:**
- Logistic Regression (baseline — interpretable, fast)
- XGBoost (production model — higher accuracy, tree-based)

**Sections:**
1. Dataset overview and class imbalance
2. Train / test split
3. Model training
4. Why not accuracy? The misleading metric problem
5. ROC curve comparison
6. Precision-Recall curve — the right metric
7. Confusion matrix with business interpretation
8. Calibration plot — are predicted probabilities trustworthy?
9. SHAP global feature importance
10. Model selection summary

In [ ]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    roc_auc_score, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, MODELS_DIR, CONFIG
from src.features.build_features import get_model_matrix

cfg = CONFIG
print('Setup complete.')

## 1. The Dataset: Understanding Class Imbalance

Before evaluating any model, we need to understand the data it was trained on. Churn prediction is a **class imbalance** problem: only a small fraction of customers churn in any given month.

**Why does this matter?** A naive model that always predicts "no churn" achieves very high accuracy — but it would be completely useless for a retention team. This is why standard accuracy is a broken metric here, and we need alternatives designed for rare-event detection.

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'churn_features.csv')
X, y = get_model_matrix(df)

print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Churned rows:   {y.sum():,} of {len(y):,} total')
print(f'Churn rate:     {y.mean():.2%}')

In [ ]:
counts = y.value_counts().sort_index()
not_churned, churned = int(counts[0]), int(counts[1])
imbalance_ratio = not_churned / churned

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'xy'}]],
    subplot_titles=('Class Distribution', 'Count Comparison')
)

fig.add_trace(go.Pie(
    labels=['Not Churned', 'Churned'],
    values=[not_churned, churned],
    marker_colors=['#2196F3', '#F44336'],
    textinfo='label+percent',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=['Not Churned', 'Churned'],
    y=[not_churned, churned],
    marker_color=['#2196F3', '#F44336'],
    text=[f'{not_churned:,}', f'{churned:,}'],
    textposition='auto',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Churn Class Imbalance in Training Data',
    height=400,
    template='plotly_white'
)
fig.show()

print(f'Imbalance ratio: {imbalance_ratio:.0f}:1 (not churned : churned)')
print(f'A model predicting no churn for everyone would be {not_churned / (not_churned + churned):.1%} accurate -- and useless.')

> **Business interpretation:** With a ~1.2% per-month churn rate, the "positive" class (churners) is extremely rare. This means:
> - Standard accuracy is meaningless — a model predicting zero churns looks 98%+ accurate
> - The model must be evaluated on **how well it finds the rare churners**, not overall correctness
> - Both models use class-weighting or `scale_pos_weight` to explicitly penalize missing a churner more than a false alarm

## 2. Train / Test Split

We use a **stratified 80/20 split**: 80% of rows go to training, 20% to the held-out test set. `stratify=y` ensures the same churn rate in both splits — without it, the test set might accidentally have no churners at all.

**Important:** Each row is a *customer-month* observation. One customer can appear in both training and test across different months. This mirrors the real use case: train on historical patterns, predict on recent behavior.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg['churn_model']['test_size'],
    random_state=cfg['churn_model']['random_state'],
    stratify=y
)

print(f'Training set: {X_train.shape[0]:,} rows -- churn rate: {y_train.mean():.2%}')
print(f'Test set:     {X_test.shape[0]:,} rows -- churn rate: {y_test.mean():.2%}')
print(f'Churn rate preserved in both splits: stratify=y is doing its job.')

## 3. Model Training

We train two models on the same split:

- **Logistic Regression (baseline):** Simple and interpretable. `class_weight='balanced'` adjusts sample weights so the minority class (churners) contributes proportionally to the loss function.
- **XGBoost:** Gradient-boosted trees. `scale_pos_weight=5` tells XGBoost to treat each churner as 5× more important than a non-churner during training — directly encoding the business priority.

We re-train here with the same hyperparameters used in the production pipeline to produce fresh evaluation artifacts (curves, matrices) for analysis.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr_model = LogisticRegression(
    max_iter=500,
    class_weight='balanced',
    random_state=cfg['data']['random_seed']
)
lr_model.fit(X_train_sc, y_train)

lr_pred = lr_model.predict(X_test_sc)
lr_prob = lr_model.predict_proba(X_test_sc)[:, 1]

lr_metrics = {
    'accuracy':  accuracy_score(y_test, lr_pred),
    'roc_auc':   roc_auc_score(y_test, lr_prob),
    'pr_auc':    average_precision_score(y_test, lr_prob),
    'precision': precision_score(y_test, lr_pred, zero_division=0),
    'recall':    recall_score(y_test, lr_pred),
    'f1':        f1_score(y_test, lr_pred)
}

print('Logistic Regression (baseline):')
for k, v in lr_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
xgb_cfg = cfg['churn_model']['xgboost']
xgb_model = XGBClassifier(
    n_estimators=xgb_cfg['n_estimators'],
    max_depth=xgb_cfg['max_depth'],
    learning_rate=xgb_cfg['learning_rate'],
    subsample=xgb_cfg['subsample'],
    colsample_bytree=xgb_cfg['colsample_bytree'],
    scale_pos_weight=xgb_cfg['scale_pos_weight'],
    eval_metric=xgb_cfg['eval_metric'],
    random_state=cfg['data']['random_seed'],
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_metrics = {
    'accuracy':  accuracy_score(y_test, xgb_pred),
    'roc_auc':   roc_auc_score(y_test, xgb_prob),
    'pr_auc':    average_precision_score(y_test, xgb_prob),
    'precision': precision_score(y_test, xgb_pred, zero_division=0),
    'recall':    recall_score(y_test, xgb_pred),
    'f1':        f1_score(y_test, xgb_pred)
}

print('XGBoost:')
for k, v in xgb_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

## 4. Why Accuracy Is a Misleading Metric

With only ~1.2% of rows being churners, a model that predicts "no churn" for every single customer achieves very high accuracy while catching zero churners. This is the **accuracy paradox** in imbalanced classification.

**The right metric for this problem is PR-AUC** (Precision-Recall Area Under Curve). It directly measures how well the model finds the rare positive class across every decision threshold — independent of how many true negatives exist.

In [ ]:
naive_pred = np.zeros(len(y_test), dtype=int)
naive_acc = accuracy_score(y_test, naive_pred)
n_churners = int(y_test.sum())
xgb_caught = int(recall_score(y_test, xgb_pred) * n_churners)
xgb_acc = xgb_metrics['accuracy']
xgb_pr_auc = xgb_metrics['pr_auc']

print('Naive model (always predicts no churn):')
print(f'  Accuracy:        {naive_acc:.2%}  <- looks impressive!')
print(f'  Churners caught: 0 of {n_churners}')
print(f'  PR-AUC:          {y_test.mean():.4f}  <- exposes the truth')
print()
print('XGBoost:')
print(f'  Accuracy:        {xgb_acc:.2%}  <- lower than naive!')
print(f'  Churners caught: {xgb_caught} of {n_churners}')
print(f'  PR-AUC:          {xgb_pr_auc:.4f}  <- dramatically more useful')
print()
print('=> Accuracy penalizes the model for catching churners (each correct flag = lower accuracy).')
print('   PR-AUC rewards it. For imbalanced problems, always report PR-AUC.')

## 5. ROC Curve

The ROC curve plots True Positive Rate (recall) vs False Positive Rate at every possible decision threshold. ROC-AUC answers: *"If I pick one churner and one non-churner at random, how often does the model rank the churner higher?"*

**Important limitation:** With heavy class imbalance, ROC-AUC is dominated by the enormous number of true negatives. Both models can score very high here even when their ability to find churners differs substantially. This is why we also need Section 6.

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_prob)
lr_roc_auc = auc(lr_fpr, lr_tpr)
xgb_roc_auc = auc(xgb_fpr, xgb_tpr)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lr_fpr, y=lr_tpr,
    name=f'Logistic Regression (AUC = {lr_roc_auc:.3f})',
    line=dict(color='#FF9800', width=2)
))

fig.add_trace(go.Scatter(
    x=xgb_fpr, y=xgb_tpr,
    name=f'XGBoost (AUC = {xgb_roc_auc:.3f})',
    line=dict(color='#2196F3', width=2)
))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    name='Random classifier (AUC = 0.500)',
    line=dict(color='gray', dash='dash', width=1.5)
))

fig.update_layout(
    title='ROC Curve: Logistic Regression vs XGBoost',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate (Recall)',
    height=500,
    template='plotly_white',
    legend=dict(x=0.55, y=0.1)
)
fig.show()

> **Key insight:** Both models have nearly identical ROC-AUC (~0.97). Looking at this chart alone, you might conclude they are equivalent. But ROC is inflated by the massive number of true negatives in our imbalanced dataset. We need the Precision-Recall curve to see which model is actually better at finding the churners we care about.

## 6. Precision-Recall Curve

The PR curve plots Precision vs Recall at every possible threshold:
- **Precision** = of the customers we flag as high risk, what fraction actually churn?
- **Recall** = of all customers who actually churn, what fraction do we flag?

The **no-skill baseline** for PR-AUC equals the positive class rate (~1.2%). A useless model lands on this line. A good model should be *far above it*.

**Unlike ROC, PR-AUC is not inflated by true negatives.** Every point on the curve measures how useful the model is for the rare class we actually care about.

In [ ]:
lr_prec, lr_rec, _ = precision_recall_curve(y_test, lr_prob)
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_prob)
baseline_rate = float(y_test.mean())
lr_ap = lr_metrics['pr_auc']
xgb_ap = xgb_metrics['pr_auc']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lr_rec, y=lr_prec,
    name=f'Logistic Regression (AP = {lr_ap:.3f})',
    line=dict(color='#FF9800', width=2)
))

fig.add_trace(go.Scatter(
    x=xgb_rec, y=xgb_prec,
    name=f'XGBoost (AP = {xgb_ap:.3f})',
    line=dict(color='#2196F3', width=2)
))

fig.add_hline(
    y=baseline_rate,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'No-skill baseline ({baseline_rate:.3f})',
    annotation_position='bottom right'
)

fig.update_layout(
    title='Precision-Recall Curve: Logistic Regression vs XGBoost',
    xaxis_title='Recall (fraction of all churners caught)',
    yaxis_title='Precision (fraction of flagged customers who actually churn)',
    height=500,
    template='plotly_white',
    legend=dict(x=0.5, y=0.9)
)
fig.show()

improvement = (xgb_ap - lr_ap) / lr_ap
print(f'XGBoost PR-AUC: {xgb_ap:.3f}')
print(f'LR PR-AUC:      {lr_ap:.3f}')
print(f'Improvement:    +{improvement:.0%} -- this gap is invisible in ROC-AUC')

> **Key finding:** XGBoost's PR-AUC is substantially higher than Logistic Regression — a gap that was invisible in the ROC curve. This gap has direct operational value: at the same recall level (same number of churners caught), XGBoost produces far fewer false alarms, meaning the CS team spends less time on customers who were never going to churn.

## 7. Confusion Matrix

At a fixed decision threshold (default: 0.5), the confusion matrix breaks predictions into four buckets. Each has a different business cost:

| | Predicted: No Churn | Predicted: Churn |
|---|---|---|
| **Actually No Churn** | True Negative (correct, leave alone) | False Positive (wasted CS outreach) |
| **Actually Churns** | False Negative **(missed = lost ARR)** | True Positive (caught, intervene) |

**False Negatives are the most costly:** a missed churner costs the company their full ARR. A False Positive only costs one CSM call. This asymmetry is why we choose a lower threshold in production.

In [ ]:
cm = confusion_matrix(y_test, xgb_pred)
tn, fp, fn, tp = cm.ravel()

fig = go.Figure(go.Heatmap(
    z=[[tn, fp], [fn, tp]],
    x=['Predicted: No Churn', 'Predicted: Churn'],
    y=['Actual: No Churn', 'Actual: Churn'],
    colorscale=[[0, '#E3F2FD'], [1, '#1565C0']],
    text=[[f'{tn}', f'{fp}'], [f'{fn}', f'{tp}']],
    texttemplate='<b>%{text}</b>',
    showscale=False,
    hovertemplate='%{y} | %{x}: %{text}<extra></extra>'
))

fig.update_layout(
    title='Confusion Matrix -- XGBoost (decision threshold = 0.5)',
    height=420,
    template='plotly_white',
    xaxis=dict(side='top'),
    font=dict(size=14)
)
fig.show()

print(f'True Positives  (churners correctly flagged):        {tp:4d}')
print(f'False Negatives (churners missed):                   {fn:4d}  <- each one = lost ARR')
print(f'False Positives (non-churners unnecessarily flagged): {fp:4d}  <- wasted CS effort')
print(f'True Negatives  (non-churners correctly left alone): {tn:4d}')
print()
prec_val = tp / (tp + fp + 1e-9)
rec_val = tp / (tp + fn + 1e-9)
print(f'Precision: {prec_val:.1%} -- of flagged customers, this fraction actually churns')
print(f'Recall:    {rec_val:.1%} -- of all churners, this fraction was caught')

> **Threshold tuning:** The default 0.5 threshold is conservative. In production, lowering it to ~0.3 would catch more churners at the cost of more false positives — a deliberate business decision based on CSM team capacity. A team that can handle 25 outreach calls per week should set the threshold to generate roughly 25 high-confidence flags per week.

## 8. Calibration Plot

Calibration answers the question: **"When the model says 70% churn probability, do 70% of those customers actually churn?"**

This matters because the Revenue Risk Score and ARR-at-risk calculations in RevIQ AI use the raw churn probability as a number. If the model says 0.80 but the actual rate is 0.30, the ARR-at-risk figures shown to executives are misleading.

A perfectly calibrated model follows the diagonal line. Points above the diagonal mean the model underestimates risk; points below mean it overestimates.

In [ ]:
xgb_frac, xgb_mean = calibration_curve(y_test, xgb_prob, n_bins=5, strategy='quantile')
lr_frac, lr_mean = calibration_curve(y_test, lr_prob, n_bins=5, strategy='quantile')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    name='Perfect calibration',
    line=dict(color='gray', dash='dash', width=1.5)
))

fig.add_trace(go.Scatter(
    x=xgb_mean, y=xgb_frac,
    name='XGBoost',
    mode='lines+markers',
    line=dict(color='#2196F3', width=2),
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=lr_mean, y=lr_frac,
    name='Logistic Regression',
    mode='lines+markers',
    line=dict(color='#FF9800', width=2),
    marker=dict(size=8)
))

fig.update_layout(
    title='Calibration Plot -- Do Predicted Probabilities Match Reality?',
    xaxis_title='Mean Predicted Probability',
    yaxis_title='Fraction of Positives (Actual Churn Rate in Bin)',
    height=450,
    template='plotly_white'
)
fig.show()

print('Interpretation:')
print('  - Points above the diagonal: model underestimates risk (actual churn > predicted)')
print('  - Points below the diagonal: model overestimates risk (actual churn < predicted)')
print('  - Note: with very few positive examples per bin, calibration curves are noisy at low probabilities')

## 9. SHAP Global Feature Importance

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score for every individual prediction. The **global importance** is the mean absolute SHAP value across all test samples — it answers:

*"Which features move churn predictions the most, on average, across all customers?"*

This goes deeper than standard XGBoost feature importance, which only counts how often a feature is used as a split. SHAP measures the **actual magnitude of impact** on each prediction, making it directly interpretable for business stakeholders.

In [ ]:
FEATURE_LABELS = {
    'logins': 'Monthly Logins',
    'active_users': 'Active Users',
    'feature_adoption': 'Feature Adoption Rate',
    'api_calls': 'API Calls',
    'logins_roll3m': 'Avg Logins (3-month)',
    'logins_trend3m': 'Login Trend (3-month)',
    'feature_adoption_roll3m': 'Avg Feature Adoption (3-month)',
    'tickets': 'Support Tickets',
    'tickets_roll3m': 'Support Tickets (3-month)',
    'sentiment': 'Support Sentiment',
    'sentiment_roll3m': 'Avg Sentiment (3-month)',
    'avg_resolution_time': 'Avg Resolution Time',
    'nps': 'Net Promoter Score',
    'health_score': 'Health Score',
    'last_touch_days': 'Days Since Last CSM Touch',
    'mrr': 'Monthly Recurring Revenue',
    'arr': 'Annual Recurring Revenue',
    'months_to_renewal': 'Months to Renewal',
    'mrr_change_pct': 'MRR Change %',
    'csm_assigned': 'CSM Assigned',
    'mrr_lag1': 'MRR (Prior Month)',
    'logins_lag1': 'Logins (Prior Month)',
    'tickets_lag1': 'Tickets (Prior Month)',
    'nps_lag1': 'NPS (Prior Month)',
    'health_score_lag1': 'Health Score (Prior Month)',
    'month': 'Month (time index)',
}

print('Computing SHAP values...')
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_test)

mean_abs_shap = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False).head(15)

mean_abs_shap.index = [FEATURE_LABELS.get(f, f) for f in mean_abs_shap.index]

fig = go.Figure(go.Bar(
    x=mean_abs_shap.values[::-1],
    y=mean_abs_shap.index[::-1],
    orientation='h',
    marker_color='#2196F3',
    text=[f'{v:.4f}' for v in mean_abs_shap.values[::-1]],
    textposition='outside'
))

fig.update_layout(
    title='SHAP Global Feature Importance -- Top 15 Churn Drivers',
    xaxis_title='Mean |SHAP Value| (average impact on churn prediction)',
    height=550,
    template='plotly_white',
    margin=dict(l=250, r=100)
)
fig.show()

print('Top 5 churn drivers:')
for feat, val in mean_abs_shap.head(5).items():
    print(f'  {feat:<35}: {val:.4f}')

## 10. Model Selection Summary

### Decision

XGBoost is the production model. The table below shows why.

### Design choices and their rationale

| Choice | Rationale |
|---|---|
| **PR-AUC as primary metric** | Accuracy is meaningless at 1.2% churn rate; ROC-AUC is inflated by true negatives |
| **XGBoost over Logistic Regression** | Higher PR-AUC; captures non-linear interactions between usage, NPS, and renewal timing |
| **scale_pos_weight=5** | Encodes the business asymmetry: missing a churner costs more than a false alarm |
| **SHAP for explainability** | Coefficient-based importance is misleading with correlated features; SHAP shows directional effects |
| **threshold = 0.5 (default)** | Conservative default; lower in production based on CS team capacity |

### Known limitations

- The model is trained on **synthetic data** — real-world performance would require validation on production churn signals
- With only ~1.2% churn rate, **calibration is noisy** — raw probabilities should be treated as relative risk scores, not absolute rates
- The **threshold is not tuned** — in production, use the PR curve to pick the threshold that matches CS team outreach capacity

In [ ]:
metrics_order = ['accuracy', 'roc_auc', 'pr_auc', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'ROC-AUC', 'PR-AUC *', 'Precision', 'Recall', 'F1 Score']
lr_vals = [f'{lr_metrics[k]:.3f}' for k in metrics_order]
xgb_vals = [f'{xgb_metrics[k]:.3f}' for k in metrics_order]
winners = [
    'Tie  (misleading for imbalanced data)',
    'Tie  (inflated by true negatives)',
    'XGBoost  (primary metric)',
    '--',
    '--',
    'XGBoost'
]

summary = pd.DataFrame({
    'Logistic Regression': lr_vals,
    'XGBoost': xgb_vals,
    'Winner / Note': winners
}, index=metric_labels)

print(summary.to_string())
print()
print('* PR-AUC is the metric used to select the production model.')